# 06 — Evaluation Analysis (Phase 6)

What this covers:
1. Load both models (LSS + zero-shot IPM)
2. Overall IoU table: LSS vs IPM
3. Per-class IoU bar chart
4. Terrain-stratified analysis: Boston vs Singapore
5. Error maps: TP/FP/FN visualisation per class
6. Side-by-side: GT vs LSS vs IPM on same sample

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils.config import CLASSES, BEV, PATHS
from utils.device import get_device

device = get_device(verbose=True)
print('Ready')

## 1. Load both models

In [ ]:
from models.lss.lss_model import LSSModel
from models.segformer.model import build_segformer_zero_shot
from train.checkpointing import load_lss, lss_path

# LSS
lss_model = LSSModel().to(device)
if lss_path().exists():
    epoch, best_miou = load_lss(lss_model, device=device)
    print(f'LSS loaded: epoch={epoch}  best_mIoU={best_miou:.4f}')
else:
    print('No LSS checkpoint — run training first')
lss_model.eval()

# IPM (zero-shot SegFormer)
seg_model = build_segformer_zero_shot().to(device)
seg_model.eval()
print('Zero-shot SegFormer loaded')

## 2. Overall IoU table: LSS vs IPM

In [ ]:
from eval.evaluate import evaluate_both
from eval.results_table import print_results_table, save_results_csv

print('Running evaluation (this takes ~2-3 min)...')
results = evaluate_both(lss_model, seg_model, split='val', device=device)
lss_r   = results['lss']
ipm_r   = results['ipm']

print_results_table(lss_r, ipm_r)

## 3. Per-class IoU bar chart

In [ ]:
cls_names = CLASSES['names'] + ['mIoU']
lss_vals  = [lss_r.get(n, 0.0) for n in cls_names]
ipm_vals  = [ipm_r.get(n, 0.0) for n in cls_names]

x     = np.arange(len(cls_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, ipm_vals, width, label='IPM (baseline)',
               color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, lss_vals, width, label='LSS (ours)',
               color='coral',    alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(cls_names)
ax.set_ylabel('IoU'); ax.set_ylim(0, 1.05)
ax.set_title('BEV Semantic Segmentation IoU: LSS vs IPM (val split)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout(); plt.show()

## 4. Terrain-stratified analysis: Boston vs Singapore

In [ ]:
from eval.terrain_analysis import evaluate_by_terrain, print_terrain_table

print('Terrain-stratified evaluation...')
lss_terrain = evaluate_by_terrain(lss_model, model_type='lss',
                                   split='val', device=device)
ipm_terrain = evaluate_by_terrain(None, model_type='ipm',
                                   seg_model=seg_model,
                                   split='val', device=device)

print_terrain_table(lss_terrain, ipm_terrain)

In [ ]:
# Terrain bar chart
cities    = ['boston', 'singapore']
city_lbls = ['Boston (non-flat)', 'Singapore (flat)']
colors    = {'lss': 'coral', 'ipm': 'steelblue'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, city, clbl in zip(axes, cities, city_lbls):
    names  = CLASSES['names'] + ['mIoU']
    lss_v  = [lss_terrain.get(city, {}).get(n, 0.0) for n in names]
    ipm_v  = [ipm_terrain.get(city, {}).get(n, 0.0) for n in names]
    x      = np.arange(len(names))
    ax.bar(x - 0.2, ipm_v, 0.35, label='IPM', color='steelblue', alpha=0.85)
    ax.bar(x + 0.2, lss_v, 0.35, label='LSS', color='coral',     alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
    ax.set_ylim(0, 1.1); ax.set_ylabel('IoU')
    ax.set_title(clbl); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Terrain-stratified IoU: LSS advantage larger in Boston (non-flat terrain)')
plt.tight_layout(); plt.show()

print('Key finding: IPM assumes flat ground — fails on Boston ramps.')
print('LSS learns depth without geometric assumptions — more robust.')

## 5. Error maps: TP/FP/FN visualisation

In [ ]:
from data.nuscenes_dataset import build_dataloader
from utils.device import move_batch
from eval.error_viz import make_all_class_error_maps

val_loader = build_dataloader('val', batch_size=1, shuffle=False)
batch      = next(iter(val_loader))
batch_dev  = {k: v.to(device) if hasattr(v,'to') else v for k,v in batch.items()}

with torch.no_grad():
    lss_logits, _ = lss_model(batch_dev['image'], batch_dev['K'],
                               batch_dev['T_cam2ego'])

lss_logits_s = lss_logits.squeeze(0).cpu()   # (C, H, W)
gt_s         = batch['bev_gt'].squeeze(0)     # (C, H, W)

lss_err_maps = make_all_class_error_maps(lss_logits_s, gt_s)

fig, axes = plt.subplots(1, CLASSES['num_classes'], figsize=(14, 4))
for ax, name in zip(axes, CLASSES['names']):
    ax.imshow(lss_err_maps[name], origin='upper')
    ax.set_title(f'LSS — {name}')
    ax.axis('off')

patches = [
    mpatches.Patch(color=(0/255,200/255,0/255),   label='TP (correct)'),
    mpatches.Patch(color=(200/255,0/255,0/255),   label='FP (false alarm)'),
    mpatches.Patch(color=(0/255,0/255,200/255),   label='FN (missed)'),
    mpatches.Patch(color=(20/255,20/255,20/255),  label='TN'),
]
axes[-1].legend(handles=patches, loc='lower right', fontsize=8)
plt.suptitle('LSS Error Maps — Green=TP, Red=FP, Blue=FN')
plt.tight_layout(); plt.show()

## 6. Side-by-side: GT vs LSS vs IPM on same sample

In [ ]:
import cv2
from ipm.pipeline import run_ipm_pipeline
from viz.colorize import colorize_bev
from data.nuscenes_loader import get_camera_data

token    = batch['sample_token'][0]
cam_data = get_camera_data(token)
bgr      = cv2.imread(str(cam_data['image_path']))
rgb      = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# IPM on same sample
ipm_result = run_ipm_pipeline(bgr, cam_data['K'],
                               sample_token=token,
                               model=seg_model, device=device)

def logits_to_label(logits_np, thresholds=(0.45, 0.20, 0.15)):
    """Per-class thresholds for visual output (lower for rare classes)."""
    probs = torch.from_numpy(logits_np).sigmoid().numpy()
    lbl   = np.zeros((probs.shape[1], probs.shape[2]), dtype=np.uint8)
    lbl[probs[0] > thresholds[0]] = 0
    lbl[probs[1] > thresholds[1]] = 1
    lbl[probs[2] > thresholds[2]] = 2
    return lbl

def channels_to_label(ch):
    lbl = np.zeros((ch.shape[-2], ch.shape[-1]), dtype=np.uint8)
    for c in range(ch.shape[0]):
        lbl[ch[c] > 0.5] = c
    return lbl

gt_lbl   = channels_to_label(batch['bev_gt'].squeeze(0).numpy())
lss_lbl  = logits_to_label(lss_logits_s.numpy())
ipm_lbl  = channels_to_label(ipm_result['bev_channels'])

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(rgb)
axes[0].set_title('Front camera'); axes[0].axis('off')

axes[1].imshow(colorize_bev(gt_lbl),  origin='upper')
axes[1].plot(100,100,'y*',markersize=10)
axes[1].set_title('BEV Ground Truth'); axes[1].axis('off')

axes[2].imshow(colorize_bev(lss_lbl), origin='upper')
axes[2].plot(100,100,'y*',markersize=10)
axes[2].set_title(f'LSS prediction\nmIoU={lss_r["mIoU"]:.3f}')
axes[2].axis('off')

axes[3].imshow(colorize_bev(ipm_lbl), origin='upper')
axes[3].plot(100,100,'y*',markersize=10)
axes[3].set_title(f'IPM prediction\nmIoU={ipm_r["mIoU"]:.3f}')
axes[3].axis('off')

patches = [
    mpatches.Patch(color=np.array([100,160,100])/255, label='drivable'),
    mpatches.Patch(color=np.array([200,80,50])/255,   label='vehicle'),
    mpatches.Patch(color=np.array([50,120,220])/255,  label='pedestrian'),
]
axes[3].legend(handles=patches, loc='lower right', fontsize=8)

plt.suptitle(f'LSS vs IPM — token: {token[:8]}...', fontsize=12)
plt.tight_layout(); plt.show()

## 7. Save results to CSV

In [ ]:
from eval.results_table import save_results_csv

csv_path = save_results_csv(lss_r, ipm_r, lss_terrain, ipm_terrain)
print(f'Saved: {csv_path}')

import csv
with open(csv_path) as f:
    rows = list(csv.DictReader(f))
print(f'Rows: {len(rows)}')
print('Columns:', list(rows[0].keys()))

## Summary

Run full evaluation from command line:
```bash
bash scripts/run_evaluation.sh
```

Run verification:
```bash
python scripts/verify_phase6.py
```

Key results to report in README:
- LSS mIoU vs IPM mIoU (overall)
- LSS Boston vs Singapore (terrain robustness)
- IPM fails on Boston ramps — LSS is more consistent
